# Post-Layer 2 Segmentation Review

Scientist-facing inspection notebook for Phase A+ window QC review.

**Important:** After backend code changes, use **Kernel → Restart Kernel and Run All** (or re-run the code cell) before **Run Review**. Otherwise Jupyter may use cached old Python modules.

**Workflow:** choose session inputs → choose frame window → choose labeled-marker QC evidence → choose link/joint scope → run review → inspect summary and tables.

**Scientific notes**
- Layer 1 labeled-marker QC is regional risk evidence, not proof of link invalidity.
- Unlabeled-marker evidence is excluded from the main Notebook UX V1.
- Layer 2 remains authoritative for solved kinematic link usability.
- DataDescriptions enrichment is optional and unverified in Phase A+.
- The Link / Joint table is a review/status table, not a recommendation engine.
- Segment export is not implemented yet.

In [1]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import ipywidgets as widgets
from IPython.display import HTML, Markdown, clear_output, display

try:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")
except Exception:
    pass

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
SRC = REPO_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import layer2_motive.segmentation.qc_events as qc_events_mod
import layer2_motive.segmentation.notebook_review as notebook_review_mod

importlib.reload(qc_events_mod)
importlib.reload(notebook_review_mod)

from layer2_motive.segmentation.load_inputs import InputLoadError
from layer2_motive.segmentation.notebook_review import (
    NOTEBOOK_QC_EVIDENCE_OPTIONS,
    audit_file_info,
    collect_output_paths,
    format_review_input_summary_markdown,
    load_audit_preview,
    load_review_outputs,
    load_summary_json,
    prepare_review_input_summary,
    prepare_scientist_link_joint_table,
    prepare_scientist_qc_event_table,
    run_review_from_notebook,
)

BACKEND_MARKER = "gap_files_parsed_v2" if hasattr(qc_events_mod, "normalize_gap_summary_events") else "legacy"

DEFAULT_LAYER1 = REPO_ROOT / "input" / "Layer1_QC" / "QC_671_T1_P1_R1"
DEFAULT_LAYER2 = REPO_ROOT / "input" / "Layer2_Kinematics" / "671_T1_P1_R1"
DEFAULT_DATADESC = (
    REPO_ROOT
    / "data_description"
    / "671_T1_P1_R1_Take 2026-01-06 03.57.12 PM_001_DataDescriptions.csv"
)

layer1_dir = widgets.Text(
    value=str(DEFAULT_LAYER1),
    description="Layer 1 session folder",
    layout=widgets.Layout(width="95%"),
)
layer2_dir = widgets.Text(
    value=str(DEFAULT_LAYER2),
    description="Layer 2 session folder",
    layout=widgets.Layout(width="95%"),
)
datadescriptions_path = widgets.Text(
    value=str(DEFAULT_DATADESC) if DEFAULT_DATADESC.exists() else "",
    description="DataDescriptions (optional)",
    layout=widgets.Layout(width="95%"),
)

start_frame = widgets.BoundedIntText(value=16000, min=0, max=500000, description="start_frame")
end_frame = widgets.BoundedIntText(value=17000, min=0, max=500000, description="end_frame")
qc_evidence = widgets.SelectMultiple(
    options=list(NOTEBOOK_QC_EVIDENCE_OPTIONS),
    value=tuple(NOTEBOOK_QC_EVIDENCE_OPTIONS),
    description="QC evidence to include",
    layout=widgets.Layout(width="360px", height="120px"),
)
export_scope = widgets.Dropdown(
    options=["core_candidate", "core_plus_review", "all_non_excluded", "all_links_audit"],
    value="core_candidate",
    description="link/joint scope",
)

run_button = widgets.Button(description="Run Review", button_style="primary")
show_audit_preview = widgets.Checkbox(value=False, description="Show first 20 audit rows (optional)")
status_output = widgets.Output()


def _validate_paths() -> str | None:
    l1 = Path(layer1_dir.value.strip())
    l2 = Path(layer2_dir.value.strip())
    if not l1.is_dir():
        return f"Layer 1 session folder not found: {l1}"
    if not l2.is_dir():
        return f"Layer 2 session folder not found: {l2}"
    dd = datadescriptions_path.value.strip()
    if dd and not Path(dd).is_file():
        return f"DataDescriptions file not found: {dd}"
    return None


def on_run_clicked(_btn):
    importlib.reload(qc_events_mod)
    importlib.reload(notebook_review_mod)
    run_review = notebook_review_mod.run_review_from_notebook
    load_outputs = notebook_review_mod.load_review_outputs
    load_summary = notebook_review_mod.load_summary_json
    prep_summary = notebook_review_mod.prepare_review_input_summary
    prep_qc = notebook_review_mod.prepare_scientist_qc_event_table
    prep_link = notebook_review_mod.prepare_scientist_link_joint_table
    fmt_summary = notebook_review_mod.format_review_input_summary_markdown
    get_paths = notebook_review_mod.collect_output_paths
    get_audit = notebook_review_mod.audit_file_info
    preview_audit = notebook_review_mod.load_audit_preview

    with status_output:
        clear_output(wait=True)
        path_error = _validate_paths()
        if path_error:
            display(HTML(f"<b style='color:red'>Validation:</b> {path_error}"))
            return
        try:
            dd_path = datadescriptions_path.value.strip() or None
            out_dir = (
                REPO_ROOT
                / "outputs"
                / "notebook_review"
                / f"{Path(layer1_dir.value).name}_{start_frame.value}_{end_frame.value}_{export_scope.value}"
            )
            result = run_review(
                layer1_dir.value,
                layer2_dir.value,
                start_frame.value,
                end_frame.value,
                qc_evidence=list(qc_evidence.value),
                export_scope=export_scope.value,
                datadescriptions=dd_path,
                out=out_dir,
            )
            outputs = load_outputs(result.out_dir)
            summary_json = load_summary(result.out_dir)
            input_summary = prep_summary(result, outputs, summary_json)
            qc_table = prep_qc(
                outputs.qc_event_display,
                qc_evidence=list(qc_evidence.value),
            )
            link_table = prep_link(outputs.layer2_link_scope)
            paths = get_paths(result.out_dir)
            audit = get_audit(result.out_dir)
            gap_status = summary_json.get("gap_files_status", {})

            if gap_status.get("gaps_over_0p5s.csv") == "present_parsed":
                display(HTML("<b style='color:green'>Backend:</b> gap/artifact files parsed (current code active)"))
            else:
                display(
                    HTML(
                        "<b style='color:orange'>Backend:</b> gap files not marked parsed — "
                        "restart kernel and re-run this cell, then Run Review again."
                    )
                )

            display(Markdown(fmt_summary(input_summary)))

            display(Markdown("### QC Event Review Table"))
            display(HTML(f"<i>{len(qc_table)} labeled-marker events in selected window/evidence.</i>"))
            display(qc_table.head(200))
            if len(qc_table) > 200:
                display(HTML("<i>Showing first 200 rows.</i>"))
            display(HTML(f"<b>Full file:</b> <code>{paths['qc_event_display']}</code>"))

            display(Markdown("### Link / Joint Review Table"))
            display(HTML(f"<i>{len(link_table)} links/joints in selected scope.</i>"))
            display(link_table)
            display(HTML(f"<b>Full file:</b> <code>{paths['layer2_link_scope']}</code>"))

            display(Markdown("### Audit / report paths"))
            display(
                HTML(
                    f"Full audit path: <code>{paths['combined_qc_events']}</code> ({audit.row_count} rows)<br>"
                    f"Review report path: <code>{paths['window_review_report']}</code><br>"
                    f"Validation JSON path: <code>{paths['window_validation_summary']}</code>"
                )
            )
            if show_audit_preview.value:
                display(Markdown("#### Optional audit preview (first 20 rows)"))
                display(preview_audit(result.out_dir, nrows=20))

            if result.warnings:
                display(Markdown("**Warnings**"))
                for warning in result.warnings:
                    display(HTML(f"- {warning}"))
        except InputLoadError as exc:
            display(HTML(f"<b style='color:red'>Input load error:</b> {exc}"))
        except ValueError as exc:
            display(HTML(f"<b style='color:red'>Review error:</b> {exc}"))


run_button.on_click(on_run_clicked)

display(
    widgets.VBox(
        [
            layer1_dir,
            layer2_dir,
            datadescriptions_path,
            widgets.HBox([start_frame, end_frame]),
            widgets.HBox([qc_evidence, export_scope]),
            run_button,
            show_audit_preview,
            status_output,
        ]
    )
)